# Exercise 3:
<b>Instructions</b>
* Keep a subset of columns and clean up column names (no spaces, newlines, etc):
    * columns related to identifying the agency
    * population, passenger trips
    * transit mode
    * at least 3 service metric variables, normalized and not normalized
* Deal with duplicates - what is the unit for each row? What is the unit for desired analysis? Should an agency appear multiple times, and if so, why?
* Aggregate at least 2 ways and show an interesting comparison, after dealing with duplicates somehow (either aggregation and/or defining what the unit of analysis is)
* Calculate weighted average after the aggregation for the service metrics
* Decide on one type of chart to visualize, and generalize it as a function
* Make charts using the function

### Helpful Hints for Functions
* Opportunities are from components that are generalizable in making a chart
* Maybe these components need the same lines of code to clean them
* You can always further define variables within a function
* You can always use f-strings within functions to make slight modifications to the parameters you pass

## Reading in my file & cleaning it up

In [1]:
import pandas as pd
import numpy as np
import shared_utils
import altair as alt
from shared_utils import altair_utils 

pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'shared_utils'

In [ ]:
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/bus_service_increase/"
FILE_NAME = "ntd_metrics_2019.csv"
df2 = pd.read_csv(f"{GCS_FILE_PATH}{FILE_NAME}")

In [ ]:
#cleaning up columns
df2.columns = df2.columns.str.strip().str.replace(' ', '_')

## Subsetting for the 3 performance metrics & agency identifiers.

<b> Just notes on the different abbreviations
* TOS: Types of Service

* MB: Driving a bus

* DO: Vehicle Maintenance 

* VOMS: vehicles operated in annual maximum service

In [ ]:
list(df2.columns)

In [ ]:
df3 = df2[['Agency', 'City', 'State', 'Legacy_NTD_ID', 'NTD_ID',
    'Primary_UZA\n_Population', 'Mode','Cost_per\n_Hour','Passengers_per_Hour','Cost_per_Passenger','TOS','Unlinked_Passenger_Trips','Total_Operating_Expenses']]

In [ ]:
df3 = df3.rename(columns = {'Primary_UZA\n_Population': 'Primary_Population', 'Cost_per\n_Hour': 'Cost_per_hour'}) #renaming columns 

In [ ]:
print(f"There are: {len(df3)} rows before any filtering")

In [ ]:
df3.head(2)

## Deal with Duplicates: What is the unit for each row? What is the unit for desired analysis? Should an agency appear multiple times, and if so, why?

* The unit for each row is the mode of transportation by agency & geography, looking at the cost per hour of operating the vehicle, how many passengers boarded per hour, total unlinked trips, cost per passenger, etc. 

* An agency should appear multiple times as some modes of transportation are contracted out to either vendors. Additionally, there are agencies in different states that have similar or even the same names.

In [ ]:
#looking at duplicates...these are just NA values. 
duplicateRowsDF = df3[df3.duplicated()]
duplicateRowsDF

In [ ]:
#dropping NA
df3 = df3.dropna()

In [ ]:
print(f"Number of unique agencies after we deduplicated: {df3.Agency.nunique()}")

In [ ]:
print(f"There are: {len(df3)} rows after  dropping duplicates")

In [ ]:
#looking at the few of the agencies that still have duplicates to see why...
df3[(df3.Agency.str.contains("Southern Teton Area Rapid Transit"))]

In [ ]:
#looking at the few of the agencies that still have duplicates to see why...some agencies have the same name in didfferent states
df3[(df3.Agency.str.contains("Union County Transit"))]

In [ ]:
#some agencies have VERY similar names in different states...The transit agency Jackson County appears in both NC and GA
df3[(df3.Agency.str.contains("Jackson County"))]

In [ ]:
#same thing, there's a Valley Transit in Arkansas and in Washington.
df3[(df3.Agency.str.contains("Valley Transit"))]

## Changing mode abbrevations 
* Mapping a few of the abbreviations to the full name for ease.

* [Transit.dot.gov's data dictionary](https://www.transit.dot.gov/ntd/national-transit-database-ntd-glossary#C)

#### Trying to map abbreviations with df.apply
Write df.apply statement lambda function.

Tiffany's code

```
df["mode_full_name"] = df.apply(lambda x: "Other" if x.mode_full_name=="" else x.mode_full_name, axis=1) #axis=1, is row

RAIL = ["LR", "HR", "CR"]
BUS = ["CB", "MB", "AR", "DR"]

def replace_modes(row):
    if RAIL in row.Mode:
        return "rail"
    elif BUS in row.Mode:
        return "bus"
    else:
        return "other"
        
df["mode_full_name"] = df.apply(lambda x: replace_modes(x), axis=1)
```

In [ ]:
df3.Mode.unique().tolist()

In [ ]:
#creating a list for all rail and all bus related modes of transportation.
RAIL = ['HR',
'DR', 'CR', 'LR',
'YR', 'SR',
'TR',
'AR']

BUS = [
 'CB',
 'MB',
 'RB',
 'FB',
 'TB','PB']

### HELP This didn't work for me, error message says "TypeError: 'in <string>' requires string as left operand, not list" but when I convert it into a string, it just puts everything in the other category.

In [ ]:
def replace_modes(row):
    if row.Mode in RAIL:
        return "Rail"
    elif row.Mode in BUS:
        return "Bus"
    else:
        return "Other"
        

In [ ]:
df3["mode_full_name_test"] = df3.apply(lambda x: replace_modes(x), axis=1) 


In [ ]:
df3.head(3)

#### Trying to map it neater 

In [ ]:
df3['Full_Mode_Name'] = np.nan

In [ ]:
df3.loc[df3.Mode.isin(['CB','MB','RB','FB','TB','PB']),  'Full_Mode_Name'] = 'Bus'
df3.loc[df3.Mode.isin(['HR','DR','CR','LR','YR','SR','TR','AR']), 'Full_Mode_Name'] = 'Rail'

In [ ]:
df3[['Full_Mode_Name']] = df3[['Full_Mode_Name']].fillna(value= 'Other')

In [ ]:
df3.Full_Mode_Name.value_counts()

## Cleaning up columns

In [ ]:
(df3.columns)

In [ ]:
#cleaning up string columns
for i in ['Agency', 'City', 'State', 'Full_Mode_Name']:
    df3[i] = df3[i].str.strip().replace(',', '').astype({i: str})

In [ ]:
df3["Cost_per_Passenger"].replace({"1,218.00": "1218"}, inplace=True)

In [ ]:
#cleaning up the numeric columns
for i in ['Primary_Population', 'Cost_per_hour', 'Passengers_per_Hour',
       'Cost_per_Passenger', 'Unlinked_Passenger_Trips', 'Total_Operating_Expenses']:
    df3[i] = df3[i].replace({'\$':''}, regex=True).replace(',','', regex= True).astype({i: float})

In [ ]:
#making sure all the data types are accurate
df3.dtypes

In [ ]:
#dropping the old mode 
df3 = df3.drop(columns = ['Mode', 'mode_full_name_test'])

In [ ]:
df3.to_csv("./df3.csv", index= False) #just exporting to CSV so I can check it out

# Aggregation & Chart Making

### First Analysis


In [ ]:
#filter out just bus for mode
bus = df3[df3["Full_Mode_Name"] == "Bus"]

In [ ]:
bus.head(2)

In [ ]:
first= bus.drop_duplicates().groupby(['State', 'Full_Mode_Name']).agg({'Unlinked_Passenger_Trips': 'sum', 'Total_Operating_Expenses': 'sum' }).reset_index()
first.sort_values(by="Unlinked_Passenger_Trips", ascending= False)

In [ ]:
first = first.nlargest(10,'Unlinked_Passenger_Trips')
first["Operating_Expenses_to_Trips"] = first.Total_Operating_Expenses.divide(first.Unlinked_Passenger_Trips)
 
first

In [ ]:
state_dropdown = first.State.unique().tolist()


In [ ]:
input_dropdown = alt.binding_select(options=state_dropdown)
selection = alt.selection_single(fields=['State'], bind=input_dropdown, name='State')

In [ ]:
#Just saving this because of the drop down menu 
#def bar_chart(df, x_col, y_col):
   # chart = (alt.Chart(df).mark_bar().encode(
           #      x=alt.X(x_col, title=x_col),
            #     y=alt.Y(y_col, title=y_col),   
              #   color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                # tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)])#.add_selection(selection).transform_filter(selection)
          #  )
    
  #  return chart

### HELP I feel like the colors are not working?

In [ ]:
def bar_chart(df, x_col, y_col):
    chart = (alt.Chart(df).mark_bar().encode(
                 x=alt.X(x_col, title=x_col),
                 y=alt.Y(y_col, title=y_col),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250)
            )
    
    return chart

In [ ]:
chart_one = bar_chart(first, 'State', 'Operating_Expenses_to_Trips')
chart_one

### Second Analysis

In [ ]:
 second = df3.groupby(['Full_Mode_Name']).agg({'Unlinked_Passenger_Trips': 'median', 'Total_Operating_Expenses': 'median' }).reset_index()

In [ ]:
second.head(2)

In [ ]:
chart_two = bar_chart(second, 'Full_Mode_Name', 'Total_Operating_Expenses')
chart_two 

In [ ]:
chart_three = bar_chart(second, 'Full_Mode_Name', 'Unlinked_Passenger_Trips')
chart_three

# Additional Feedback

In [ ]:
#Leave DF3 as is and create one for each mode category.
rail = df3[df3["Full_Mode_Name"] == "Rail"]
other = df3[df3["Full_Mode_Name"] == "Other"]

In [ ]:
print(f"original number of rows: {len(bus)}")

In [ ]:
print(f"original number of rows: {len(rail)}")

In [ ]:
print(f"original number of rows: {len(other)}")

In [ ]:
#write a function
def clean_data(df):
    df = df[df.State.isin(['CA','NY','MI','NE','MN'])]
    df["Operating_Cost_Per_Trip"] = df.Total_Operating_Expenses.divide(df.Unlinked_Passenger_Trips)
    return df
 

In [ ]:
rail1 = clean_data(rail)
bus1 = clean_data(bus)
other1 = clean_data(other)

### Creating a 2nd bar chart with new features

In [ ]:
#write second function for making charts with title
def bar_chart2(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    transit_mode = df['Full_Mode_Name'].iloc[0] #just looking at row one
    chart_title = f"{y_col_stripped} by State for Mode: {transit_mode.title()}" 
    chart = (alt.Chart(df).mark_bar().encode(
                 x=alt.X(x_col, title=x_col),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [ ]:
bar_chart2(bus1, "State", "Total_Operating_Expenses")

### Creating a bar graphs line by line having trouble saving jupyter notebook when I run all the lines creating graphs.

In [ ]:
bar_chart2(bus1, "State", "Total_Operating_Expenses")
bar_chart2(bus1, "State", "Unlinked_Passenger_Trips")
bar_chart2(bus1, "State", "Operating_Cost_Per_Trip")
bar_chart2(rail1, "State", "Total_Operating_Expenses")
bar_chart2(rail1, "State", "Unlinked_Passenger_Trips")
bar_chart2(rail1, "State", "Operating_Cost_Per_Trip")
bar_chart2(other1, "State", "Total_Operating_Expenses")
bar_chart2(other1, "State", "Unlinked_Passenger_Trips")
bar_chart2(other1, "State", "Operating_Cost_Per_Trip")

### Creating all 9 charts with a loop: for some reason once I run the cell with the loop, I can't save my jupyter notebook with all the 9 graphs.  I changed it to a markdown cell.

In [ ]:
subset_states = ['CA','NY','MI','NE','MN']
modes = ["Rail", "Bus", "Other"]

In [ ]:
df3["Operating_Expenses_to_Trips"] = df3.Total_Operating_Expenses.divide(df3.Unlinked_Passenger_Trips)

In [ ]:
for m in modes:
    bar_chart2(df3[(df3.Full_Mode_Name==m) & 
                (df3.State.isin(subset_states))], 
                "State", "Total_Operating_Expenses")

    bar_chart2(df3[(df3.Full_Mode_Name==m) & 
                (df3.State.isin(subset_states))], 
                "State", "Unlinked_Passenger_Trips")

    bar_chart2(df3[(df3.Full_Mode_Name==m) & 
               (df3.State.isin(subset_states))], 
               "State", "Operating_Expenses_to_Trips")

In [ ]:
# sample function to combine clean data and making viz
def make_viz(df, mode):
    df = clean_data(df[(df.Full_Mode_Name==mode) & (df.State.isin(subset_states))])
    
    for metric in ["Total_Operating_Expenses", 
                   "Unlinked_Passenger_Trips", 
                   "Operating_Expenses_to_Trips"]:
        bar_chart2(df, "State", metric)

    
    
for m in modes:
    make_viz(df, m)
